# Session 7 — Plotting with Matplotlib and Seaborn

**Goal of this session:** treat the figure as part of the science, not as decoration added at the end.

*Python for Neuroscience, session 7 of 12.*

## Why this matters

Your figure is the part of your work that people actually look at. A reviewer will spend thirty seconds on your plot and much less on your methods section.

We have been plotting since session 2 without explaining any of it. Ten minutes now and you will know what every line does.

## The anatomy of a figure

Two objects matter. The `fig` is the canvas. The `ax` is the coordinate system you draw inside. `plt.subplots()` hands you both.

Everything you want to change is a method on the `ax`.

In [ ]:
import numpy as np


def generate_toy_signal(duration=2.0, sampling_rate=500.0, noise_level=0.5,
                        freq=10.0, amplitude=1.0, seed=0):
    """A toy oscillatory signal: one sine wave plus white noise.

    This is not a recording. It is a stand-in that behaves enough like an
    alpha rhythm to practise on. Returns the time axis and the signal.
    """
    rng = np.random.default_rng(seed)
    t = np.arange(0, duration, 1 / sampling_rate)
    signal = amplitude * np.sin(2 * np.pi * freq * t)
    signal = signal + noise_level * rng.standard_normal(t.size)
    return t, signal

In [ ]:
import matplotlib.pyplot as plt

t, signal = generate_toy_signal(duration=2.0, sampling_rate=500.0, noise_level=0.5)

fig, ax = plt.subplots(figsize=(11, 4))     # canvas and axes
ax.plot(t, signal, color="#2b6cb0", linewidth=1, label="toy signal")
ax.set_xlabel("time (s)", fontsize=13)
ax.set_ylabel("amplitude (a.u.)", fontsize=13)
ax.set_title("A properly labelled figure", fontsize=15)
ax.legend(fontsize=11)
ax.tick_params(labelsize=12)
plt.tight_layout()
plt.show()

`figsize` is in inches, width first. `tight_layout()` stops your labels being cut off, and you should just always call it.

## Bad plot, good plot

Same data, twice. This is the single most useful slide in the course.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# the version people actually produce under deadline
axes[0].plot(t, signal)
axes[0].set_title("before")

# the version that survives review
axes[1].plot(t, signal, color="#2b6cb0", linewidth=1)
axes[1].set_xlabel("time (s)", fontsize=13)
axes[1].set_ylabel("amplitude (a.u.)", fontsize=13)
axes[1].set_title("after", fontsize=14)
axes[1].tick_params(labelsize=11)
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

The left panel has no units, no axis labels, and a default line weight that will disappear when the figure is shrunk into a column. Nothing about it is wrong. It is just unreadable, which amounts to the same thing.

`spines[["top", "right"]].set_visible(False)` removes the box around the plot. Small change, cleaner result.

## Histograms

A line plot shows you the signal over time. A histogram throws time away and shows you the distribution of values instead. Both are useful, and they answer different questions.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(signal, bins=40, color="#2b6cb0", edgecolor="white")
ax.set_xlabel("amplitude (a.u.)", fontsize=13)
ax.set_ylabel("count", fontsize=13)
ax.set_title("Distribution of signal amplitude", fontsize=15)
ax.tick_params(labelsize=12)
plt.tight_layout()
plt.show()

Notice the shape. A noisy sine wave does not give you a bell curve, it gives you something with weight at both extremes, because a sine spends most of its time near its peaks rather than near zero.

## Seaborn for comparisons

Seaborn sits on top of matplotlib and knows about DataFrames. For anything where you are comparing groups, it saves you real time.

Let's make two conditions out of our signal generator: one with a strong oscillation, one with a weak one, twenty segments each.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

rows = []
for i in range(20):
    _, strong = generate_toy_signal(duration=1.0, amplitude=1.2, seed=i)
    _, weak = generate_toy_signal(duration=1.0, amplitude=0.5, seed=100 + i)
    rows.append({"condition": "A", "amplitude": strong.std()})
    rows.append({"condition": "B", "amplitude": weak.std()})

amplitudes = pd.DataFrame(rows)
amplitudes.groupby("condition")["amplitude"].describe().round(3)

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.2)

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=amplitudes, x="condition", y="amplitude",
            palette=["#2b6cb0", "#a0aec0"], hue="condition", legend=False, ax=ax)
sns.stripplot(data=amplitudes, x="condition", y="amplitude",
              color="black", size=4, alpha=0.6, ax=ax)
ax.set_xlabel("condition")
ax.set_ylabel("signal amplitude (SD)")
ax.set_title("Amplitude by condition")
plt.tight_layout()
plt.show()

The box shows the median and the middle half of the data. The dots are the individual segments, drawn on top, and adding them is almost always worth it. A box plot alone can hide that your twenty points are really two clusters of ten.

## Try it yourself

Swap `sns.boxplot` for `sns.violinplot` and look at the difference. The violin shows the full shape of the distribution, which is more honest and takes more space.

**Next session:** filtering, and getting a signal back out of noise.